# FHVHV EDA: data quality

Goal: find out whether the data makes sense before building anything on top of it.
Every problem found here becomes a cleaning rule or a test in the dbt staging layer.

Works on **one month** (fits easily in RAM). Change `MONTH` to check others, e.g. `"2024-01"`.

## 1. Setup and load

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

data_dir = Path("datasets")
MONTH = "2025-01"

df = pd.read_parquet(data_dir / f"fhvhv_tripdata_{MONTH}.parquet")
zones = pd.read_csv(data_dir / "taxi_zone_lookup.csv")

n = len(df)
print(f"{MONTH}: {n:,} rows x {df.shape[1]} columns")

### Helper: register each check

`check(name, mask)` prints how many rows match a condition and stores the result, so section 11 can summarize everything in one table.

In [ ]:
checks = {}

def check(name: str, mask: pd.Series) -> None:
    count = int(mask.sum())
    checks[name] = count
    print(f"{name:50} {count:>12,}  {count / n:8.3%}")

## 2. Overview

Basic statistics of the numeric and datetime columns. Look at `min` and `max`: negatives and absurd values show up here first.

In [ ]:
df.describe().T

## 3. Null values

In [ ]:
nulls = df.isna().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
nulls.to_frame("nulls").assign(pct=lambda x: x["nulls"] / n * 100)

In [ ]:
# Are the nulls concentrated in one company? (% of each company's trips that are null)
(df[nulls.index].isna()
    .groupby(df["hvfhs_license_num"]).mean()
    .mul(100).rename(index={"HV0003": "Uber", "HV0005": "Lyft"}))

## 4. Domains of the text columns

License codes from the TLC data dictionary: `HV0002` Juno, `HV0003` Uber, `HV0004` Via, `HV0005` Lyft.

In [ ]:
LICENSES = {"HV0002": "Juno", "HV0003": "Uber", "HV0004": "Via", "HV0005": "Lyft"}

(df["hvfhs_license_num"].value_counts(dropna=False)
    .to_frame("trips")
    .assign(company=lambda x: x.index.map(LICENSES), pct=lambda x: x["trips"] / n * 100))

In [ ]:
TEXT_COLS = ["dispatching_base_num", "originating_base_num",
             "shared_request_flag", "shared_match_flag", "access_a_ride_flag",
             "wav_request_flag", "wav_match_flag"]

for col in TEXT_COLS:
    counts = df[col].value_counts(dropna=False)
    print(f"{col}:")
    print((counts / n * 100).round(2).to_string(), "\n")

In [ ]:
# Shared rides: a match without a request should not happen
pd.crosstab(df["shared_request_flag"], df["shared_match_flag"], margins=True)

## 5. Timestamps

Expected order: `request` ≤ `on_scene` ≤ `pickup` ≤ `dropoff`, and the pickup should fall inside the file's month.

In [ ]:
month_start = pd.Timestamp(f"{MONTH}-01")
month_end = month_start + pd.offsets.MonthBegin(1)

check("pickup outside the file's month",
      (df["pickup_datetime"] < month_start) | (df["pickup_datetime"] >= month_end))
check("dropoff before pickup", df["dropoff_datetime"] < df["pickup_datetime"])
check("dropoff equal to pickup", df["dropoff_datetime"] == df["pickup_datetime"])
check("pickup before request", df["pickup_datetime"] < df["request_datetime"])
check("on_scene before request", df["on_scene_datetime"] < df["request_datetime"])
check("pickup before on_scene", df["pickup_datetime"] < df["on_scene_datetime"])

In [ ]:
# Where do the out-of-month pickups fall?
outside = df.loc[(df["pickup_datetime"] < month_start) | (df["pickup_datetime"] >= month_end), "pickup_datetime"]
outside.dt.to_period("M").value_counts().sort_index().to_frame("trips")

## 6. Is `trip_time` consistent with the timestamps?

`trip_time` (seconds) should match `dropoff - pickup`.

In [ ]:
computed = (df["dropoff_datetime"] - df["pickup_datetime"]).dt.total_seconds()
diff = (df["trip_time"] - computed).abs()

print(diff.describe(percentiles=[0.5, 0.9, 0.99, 0.999]).to_string(), "\n")
check("trip_time differs > 60 s from dropoff - pickup", diff > 60)

## 7. Impossible or suspicious values

In [ ]:
MONEY_COLS = [c for c in ["base_passenger_fare", "tolls", "bcf", "sales_tax", "congestion_surcharge",
                          "airport_fee", "tips", "driver_pay", "cbd_congestion_fee"] if c in df.columns]

for col in MONEY_COLS:
    check(f"{col} < 0", df[col] < 0)

In [ ]:
hours = df["trip_time"] / 3600
speed_mph = df["trip_miles"] / hours.where(hours > 0)

check("trip_miles == 0", df["trip_miles"] == 0)
check("trip_miles > 100", df["trip_miles"] > 100)
check("trip_time == 0", df["trip_time"] == 0)
check("trip_time > 4 h", df["trip_time"] > 4 * 3600)
check("base_passenger_fare == 0", df["base_passenger_fare"] == 0)
check("base_passenger_fare > $500", df["base_passenger_fare"] > 500)
check("average speed > 80 mph", speed_mph > 80)

## 8. Distributions and outliers

Percentiles show where the "normal" range ends. Compare p99 with p99.9 and max: a big jump means a long tail of outliers.

In [ ]:
cols = ["trip_miles", "trip_time", "base_passenger_fare", "driver_pay", "tips", "tolls"]
q = df[cols].quantile([0, 0.01, 0.25, 0.5, 0.75, 0.99, 0.999, 1]).T
q.columns = ["min", "p1", "p25", "p50", "p75", "p99", "p99.9", "max"]
q

## 9. Zones

Every `PULocationID` / `DOLocationID` should exist in `taxi_zone_lookup.csv`. IDs 264 (Unknown) and 265 (Outside of NYC) are valid but carry no location.

In [ ]:
valid_ids = set(zones["LocationID"])
UNKNOWN_IDS = [264, 265]

check("PULocationID not in zone lookup", ~df["PULocationID"].isin(valid_ids))
check("DOLocationID not in zone lookup", ~df["DOLocationID"].isin(valid_ids))
check("PULocationID unknown / outside NYC", df["PULocationID"].isin(UNKNOWN_IDS))
check("DOLocationID unknown / outside NYC", df["DOLocationID"].isin(UNKNOWN_IDS))

never_used = valid_ids - set(df["PULocationID"].unique()) - set(df["DOLocationID"].unique())
print(f"\nZones with no pickup or dropoff this month: {sorted(never_used)}")

In [ ]:
# Top 10 pickup zones, with names from the lookup
(df["PULocationID"].value_counts().head(10)
    .rename_axis("LocationID").reset_index(name="trips")
    .merge(zones, on="LocationID"))

## 10. Duplicates

There is no trip ID, so a duplicate is defined as the same company, timestamps, zones and distance.

In [ ]:
KEY = ["hvfhs_license_num", "request_datetime", "pickup_datetime", "dropoff_datetime",
       "PULocationID", "DOLocationID", "trip_miles"]
check("duplicate trips (same key)", df.duplicated(subset=KEY))

## 11. Summary of all checks

Rows affected by each check. These become the rules and tests of the dbt staging model.

In [ ]:
summary = pd.Series(checks, name="rows").to_frame()
summary["pct"] = summary["rows"] / n * 100
summary.sort_values("rows", ascending=False)

## 12. Findings (2025-01) → dbt staging rules

Overall the data is **very clean**: no pickups outside the month, no dropoff before pickup, `trip_time` always matches the timestamps, every zone ID exists in the lookup, and only 3 duplicates.

| Finding | Rows | Interpretation | Proposed rule |
|---|---|---|---|
| `on_scene_datetime` and `originating_base_num` null | 24.7% | Lyft does not report them (99.7% of Lyft trips are null, Uber 0%) | **Keep** the rows. Never drop nulls blindly here, it would remove almost all Lyft trips |
| Dropoff zone 264/265 | 4.0% | Trips ending outside NYC. Valid | Keep. Map to "Outside NYC" in `dim_zone` |
| Pickup before request | 1.0% | Hypothesis: scheduled/prearranged rides. Not confirmed | Keep, add a flag `is_scheduled_suspect` |
| On scene before request | 0.9% | Same hypothesis | Same flag |
| `base_passenger_fare == 0` | 0.07% | Promotions or cancellations? | Keep, flag |
| `base_passenger_fare < 0` / `driver_pay < 0` | 2,427 / 313 | Probably adjustments or refunds | Filter out of the fact table (or move to an adjustments table) |
| `trip_miles == 0` | 0.01% | Cancelled or GPS issue | Flag |
| Extreme outliers (507 miles, 8 h, $1,420) | < 0.01% | p99.9 is reasonable (52 mi, 1.5 h, $198) | Flag above p99.9, do not delete |
| Shared match without shared request | 424 | Inconsistent flags | dbt test with a warning |
| Duplicates (same key) | 3 | Real duplicates | Deduplicate in staging |

**Next:** run this notebook with `MONTH = "2024-01"` and a summer month to see whether the findings hold across time.